# Leaf Segmentation: RGB PCA vs. Excess Green Index (ExG)

This notebook compares two different segmentation approaches on the RGB color space:
1. **Standardized PCA (without Polarity Correction)**
2. **Excess Green Index (ExG)**

### Pipeline Steps for both approaches:
- **Feature Extraction**: Either project standardized RGB channels onto the first principal component (PC1), or compute the Excess Green Index ($2g - r - b$ using normalized chromaticities).
- **Otsu's Thresholding**: Binarize the scaled 0-255 feature map.
- **Morphological Cleaning**: Apply Opening followed by Closing using a 5x5 elliptical kernel.
- **Geometric Filtering**: Filter out background noise (like hydroponic channels) using area, aspect ratio, and width-spanning constraints.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

%matplotlib inline

### Load Representative Image

We load a representative image from the `veg_2` category (`leaf-image/veg_2/IMG_7995.JPG`).

In [ ]:
img_path = 'leaf-image/veg_2/IMG_7995.JPG'

img_bgr = cv2.imread(img_path)
if img_bgr is None:
    raise FileNotFoundError(f"Could not locate the image at: {img_path}")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.title("Original RGB Image (veg_2)")
plt.axis('off')
plt.show()

### Approach 1: Standardized PCA (No Polarity Correction)

In [ ]:
print("========================================")
print("Approach 1: Standardized PCA (No Polarity Correction)")
print("========================================")

# 1. Split channels
r, g, b = cv2.split(img_rgb)

# 2. Standardize channels before PCA to prevent lightness scale dominance
flats = []
for ch in (r, g, b):
    ch_f = ch.astype(np.float32)
    std = ch_f.std()
    ch_norm = (ch_f - ch_f.mean()) / (std if std > 1e-8 else 1.0)
    flats.append(ch_norm.reshape(-1, 1))
features = np.hstack(flats)

# 3. Run PCA
mean, eigenvectors, eigenvalues = cv2.PCACompute2(features, mean=None)
projected = cv2.PCAProject(features, mean, eigenvectors)
pc1 = projected[:, 0].reshape(r.shape)

# Scale to 0-255 (No Polarity Correction)
pc1_min = pc1.min()
pc1_max = pc1.max()
pc1_scaled = np.uint8(255 * (pc1 - pc1_min) / (pc1_max - pc1_min + 1e-8))

# 4. Apply Otsu's Thresholding
ret, thresh = cv2.threshold(pc1_scaled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"[PCA] Otsu threshold value: {ret}")

# 5. Morphological Cleaning
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)

# 6. Post-processing: Geometric Contour Filtering
contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
filtered_mask = np.zeros_like(cleaned)
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 100 or area > 150000:
        continue
    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = float(w) / h
    if w > 0.8 * img_rgb.shape[1] or aspect_ratio > 4.0 or aspect_ratio < 0.25:
        continue
    cv2.drawContours(filtered_mask, [cnt], -1, 255, -1)
cleaned = filtered_mask

# Extract final filtered contours
final_contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Draw contours on RGB image copy
img_contour = img_rgb.copy()
cv2.drawContours(img_contour, final_contours, -1, (0, 255, 0), 2)

# Plot step results side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(pc1_scaled, cmap='gray')
axes[0].set_title("PC1 Projection (PCA)")
axes[0].axis('off')

axes[1].imshow(cleaned, cmap='gray')
axes[1].set_title("Filtered Binary Mask (PCA)")
axes[1].axis('off')

axes[2].imshow(img_contour)
axes[2].set_title(f"Green Contour Overlay (PCA) - Detected: {len(final_contours)}")
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Approach 2: Excess Green Index (ExG)

In [ ]:
print("========================================")
print("Approach 2: Excess Green Index (ExG)")
print("========================================")

# 1. Split channels
r, g, b = cv2.split(img_rgb)

# 2. Calculate normalized chromaticities: r = R/(R+G+B), g = G/(R+G+B), b = B/(R+G+B)
r_f = r.astype(np.float32)
g_f = g.astype(np.float32)
b_f = b.astype(np.float32)
total = r_f + g_f + b_f
total[total == 0] = 1.0

r_norm = r_f / total
g_norm = g_f / total
b_norm = b_f / total

# 3. Compute Excess Green Index: ExG = 2*g - r - b
exg = 2.0 * g_norm - r_norm - b_norm

# Scale ExG to 0-255 range
exg_min = exg.min()
exg_max = exg.max()
exg_scaled = np.uint8(255 * (exg - exg_min) / (exg_max - exg_min + 1e-8))

# 4. Apply Otsu's Thresholding
ret, thresh = cv2.threshold(exg_scaled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"[ExG] Otsu threshold value: {ret}")

# 5. Morphological Cleaning
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)

# 6. Post-processing: Geometric Contour Filtering
contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
filtered_mask = np.zeros_like(cleaned)
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 100 or area > 150000:
        continue
    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = float(w) / h
    if w > 0.8 * img_rgb.shape[1] or aspect_ratio > 4.0 or aspect_ratio < 0.25:
        continue
    cv2.drawContours(filtered_mask, [cnt], -1, 255, -1)
cleaned = filtered_mask

# Extract final filtered contours
final_contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Draw contours on RGB image copy
img_contour = img_rgb.copy()
cv2.drawContours(img_contour, final_contours, -1, (0, 255, 0), 2)

# Plot step results side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(exg_scaled, cmap='gray')
axes[0].set_title("Excess Green Index (ExG)")
axes[0].axis('off')

axes[1].imshow(cleaned, cmap='gray')
axes[1].set_title("Filtered Binary Mask (ExG)")
axes[1].axis('off')

axes[2].imshow(img_contour)
axes[2].set_title(f"Green Contour Overlay (ExG) - Detected: {len(final_contours)}")
axes[2].axis('off')

plt.tight_layout()
plt.show()